In [27]:
from pathlib import Path
from typing import Dict

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from neuralhydrology.datasetzoo import get_dataset, camelsus
from neuralhydrology.datautils.utils import load_scaler
from neuralhydrology.modelzoo.cudalstm import CudaLSTM
from neuralhydrology.modelzoo.shm import SHM
from neuralhydrology.nh_run import start_run, eval_run
from neuralhydrology.utils.config import Config
from neuralhydrology.training.basetrainer import BaseTrainer
from neuralhydrology.modelzoo.cfe_modules import get_pet

### Train the LSTM
To start, let's train an lstm for a single basin. If you're curious this is for the Narraguagus River with flow as measured at Cherryfield, Maine. I chose this for consistency with `examples/05-Inspecting-LSTMs`

In [28]:
config_file = Path("1_basin.yml")

In [ ]:
config_file = Path("1_basin.yml")
# by default we assume that you have at least one CUDA-capable NVIDIA GPU or MacOS with Metal support
if torch.cuda.is_available() or torch.backends.mps.is_available():
     start_run(config_file=config_file)

 # fall back to CPU-only mode
else:
     start_run(config_file=config_file, gpu=-1)

### Load up trained model
The results of training (model weights, metadata, and optimizer-related data) are saved in `runs`. Let's load them up.

In [4]:
run_dir = Path("runs/test_run_1203_222017")  # this value comes from the output of the above command
#!ls $run_dir/model_epoch* | tail -n 3

### Load up trained model
Let's create a new instance of the neural network and then load the trained weights into it.

In [5]:
cudalstm_config = Config(config_file)

# create a new model instance with random weights
cuda_lstm = CudaLSTM(cfg=cudalstm_config)

# load the trained weights into the new model. 
model_path = run_dir / 'model_epoch030.pt'
model_weights = torch.load(str(model_path), map_location='cpu')  # load the weights from the file, creating the weight tensors on CPU
cuda_lstm.load_state_dict(model_weights)  # set the new model's weights to the values loaded from file
cuda_lstm

CudaLSTM(
  (embedding_net): InputLayer(
    (statics_embedding): Identity()
    (dynamics_embeddings): ModuleList(
      (0): Identity()
    )
  )
  (lstm): LSTM(5, 20)
  (dropout): Dropout(p=0.4, inplace=False)
  (head): Regression(
    (net): Sequential(
      (0): Linear(in_features=20, out_features=1, bias=True)
    )
  )
)

### Configuring SHM
Now, let's initialize an SHM model.

In [ ]:
shm_config_file = Path("shm_config.yml")
shm_config = Config(shm_config_file)
shm = SHM(cfg=shm_config)
seq_len = cudalstm_config.seq_length

730


### Fetch the data
Lets instantiate a dataloader containing the data we want

In [ ]:
trainer = BaseTrainer(cfg = Config(config_file))
trainer.initialize_training()
loader = trainer.loader
itable = iter(loader)
for i in range(seq_len):
    data_point = next(itable)


2026-04-10 09:13:52,463: Logging to /home/user/Coding/neuralhydrology-dcfe/examples/09-Data-Assimilation/runs/test_run_1004_091352/output.log initialized.
2026-04-10 09:13:52,464: ### Folder structure created at /home/user/Coding/neuralhydrology-dcfe/examples/09-Data-Assimilation/runs/test_run_1004_091352
2026-04-10 09:13:52,465: ### Run configurations for test_run
2026-04-10 09:13:52,465: experiment_name: test_run
2026-04-10 09:13:52,465: train_basin_file: dry_basin.txt
2026-04-10 09:13:52,466: validation_basin_file: dry_basin.txt
2026-04-10 09:13:52,466: test_basin_file: dry_basin.txt
2026-04-10 09:13:52,466: train_start_date: 1999-10-01 00:00:00
2026-04-10 09:13:52,467: train_end_date: 2008-09-30 00:00:00
2026-04-10 09:13:52,467: validation_start_date: 1980-10-01 00:00:00
2026-04-10 09:13:52,468: validation_end_date: 1989-09-30 00:00:00
2026-04-10 09:13:52,468: test_start_date: 1989-10-01 00:00:00
2026-04-10 09:13:52,470: test_end_date: 1999-09-30 00:00:00
2026-04-10 09:13:52,470: d

In [ ]:
data_point = next(itable)

In [ ]:
#get data not shuffled by directly getting a DataLoader 
scaler = load_scaler(run_dir)
ds = get_dataset(cfg = Config(config_file), is_train = False, basin = "10258500", period = 'test', scaler=scaler)
ordered_data = DataLoader(ds, batch_size= 1, shuffle=False, num_workers= 8, collate_fn=ds.collate_fn)
itable = list(ordered_data)

In [ ]:
data_point = itable[-366] # each index backwards moves the start date back one day from 1998-10-01

In [19]:
# undo scaling
scaler = load_scaler(run_dir)
raw_x_d = {}
for feature, scaled in data_point['x_d'].items(): # key, value in dict
    center = float(scaler['xarray_feature_center'][feature].values)
    scale = float(scaler['xarray_feature_scale'][feature].values)
    raw_tensor = scaled * scale + center 
    raw_x_d[feature] = raw_tensor 

print("Scaled tmax range:", data_point['x_d']['tmax(C)'].min(), "to", data_point['x_d']['tmax(C)'].max())

center = float(scaler['xarray_feature_center']['QObs(mm/d)'].values)
scale = float(scaler['xarray_feature_scale']['QObs(mm/d)'].values)
raw_y = data_point['y'] * scale + center  

# Create a new dict with raw data, same structure as data_point
raw_data_point = {
    'x_d': raw_x_d,  
    'y': raw_y,    
    'date': data_point['date'], 
}

print("Raw tmax range:", raw_data_point['x_d']['tmax(C)'].min(), "to", raw_data_point['x_d']['tmax(C)'].max())  


Scaled tmax range: tensor(-2.2558) to tensor(1.8097)
Raw tmax range: tensor(-11.4000) to tensor(31.3438)


In [ ]:
print(data_point.keys())

In [ ]:
print(data_point['x_d'].keys())

In [ ]:
print(data_point['x_d']['tmax(C)'].shape)

In [ ]:
print(data_point['x_d']['tmax(C)'][0:5,0:2,:])

In [20]:
print(data_point['y'].shape)
print(data_point['y'])

torch.Size([1, 730, 1])
tensor([[[-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.1045],
         [-0.104

In [21]:
print(data_point['date'].shape)
print(data_point['date'])


(1, 730)
[['1999-10-01T00:00:00.000000' '1999-10-02T00:00:00.000000'
  '1999-10-03T00:00:00.000000' '1999-10-04T00:00:00.000000'
  '1999-10-05T00:00:00.000000' '1999-10-06T00:00:00.000000'
  '1999-10-07T00:00:00.000000' '1999-10-08T00:00:00.000000'
  '1999-10-09T00:00:00.000000' '1999-10-10T00:00:00.000000'
  '1999-10-11T00:00:00.000000' '1999-10-12T00:00:00.000000'
  '1999-10-13T00:00:00.000000' '1999-10-14T00:00:00.000000'
  '1999-10-15T00:00:00.000000' '1999-10-16T00:00:00.000000'
  '1999-10-17T00:00:00.000000' '1999-10-18T00:00:00.000000'
  '1999-10-19T00:00:00.000000' '1999-10-20T00:00:00.000000'
  '1999-10-21T00:00:00.000000' '1999-10-22T00:00:00.000000'
  '1999-10-23T00:00:00.000000' '1999-10-24T00:00:00.000000'
  '1999-10-25T00:00:00.000000' '1999-10-26T00:00:00.000000'
  '1999-10-27T00:00:00.000000' '1999-10-28T00:00:00.000000'
  '1999-10-29T00:00:00.000000' '1999-10-30T00:00:00.000000'
  '1999-10-31T00:00:00.000000' '1999-11-01T00:00:00.000000'
  '1999-11-02T00:00:00.000000' 

In [ ]:
shm_parameters = {
            "dd": torch.tensor(5.0, device="cpu", dtype=torch.float32),
            "f_thr": torch.tensor(60.0, device="cpu", dtype=torch.float32), # was 10
            "sumax": torch.tensor(100.0, device="cpu", dtype=torch.float32), # was 25
            "beta": torch.tensor(5.0, device="cpu", dtype=torch.float32),
            "perc": torch.tensor(0.5, device="cpu", dtype=torch.float32),
            "kf": torch.tensor(15.0, device="cpu", dtype=torch.float32), # was 3
            "ki": torch.tensor(50.0, device="cpu", dtype=torch.float32), #was 5
            "kb": torch.tensor(300.0, device="cpu", dtype=torch.float32), #was 15
        }

In [ ]:
# for loop

pt = raw_data_point #ordered_data[1]
dates = pt['date'] #[1,365]
ys = pt["y"] #[1,365,1]
x_d = pt['x_d'] # dict of forcing params mapped to [1,365,1]

# intitialize states for shm, total guess??
ss, sf, su, si, sb = shm.initialize_states(batch_size=1, device = 'cpu')
x = torch.tensor([ss,sf,su,si,sb])
x_history = np.zeros((6,730)) # to plot, = [ss, sf, su, si, sb, z_minus, ]

for j in range(ys.shape[1]): #each day
    date = dates[0, j]
    y = ys[0 ,j, 0] # true observed outflow not needed but could compare
    pet = get_pet.daily_pet_jensen2016(
                T_avg=(x_d['tmax(C)'][0, j,0] + x_d['tmin(C)'][0, j,0])/2,
                S_rad= x_d['srad(W/m2)'][0,j,0],
            )
    x_conceptual_timestep = np.stack([x_d["prcp(mm/day)"][0,j,0], pet, x_d["tmin(C)"][0,j,0], x_d["tmax(C)"][0,j,0]])
    x_conceptual_timestep = torch.tensor(x_conceptual_timestep, dtype = torch.float32, device = "cpu").unsqueeze(dim = 0) # size [1,4] row vector
    x[0], x[1], x[2], x[3], x[4], z_minus = shm.timestep_shm(x[0], x[1], x[2], x[3], x[4], shm_parameters, x_conceptual_timestep, device = x_conceptual_timestep.device)
    for i in range(4):
        x_history[i,j] = x[i]
    x_history[5,j] = z_minus

print(x)        

In [ ]:
cuda_lstm.eval()
with torch.no_grad():
    z = cuda_lstm(data_point)
print(z['y_hat'].shape)
z_plus_scaled = z['y_hat'][0, :, 0] 

# Rescale back to og
center = float(scaler['xarray_feature_center']['QObs(mm/d)'].values)
scale = float(scaler['xarray_feature_scale']['QObs(mm/d)'].values)
z_plus= z_plus_scaled * scale + center

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)  # (rows, cols, index)
plt.plot(x_d["prcp(mm/day)"][0,:,0],label="precip",linewidth=2.0)
T_avg=(x_d['tmax(C)'][0, :,0] + x_d['tmin(C)'][0, :,0])/2
plt.plot(T_avg, label="av. Temp (in C)", linewidth=2.0)
plt.title('Meteorogical Forcings')
plt.legend()

plt.subplot(1, 2, 2) 
plt.plot(x_history[0,:],label='snow',linewidth=2.0)
plt.plot(x_history[1,:],label='fast-flow',linewidth=2.0)
plt.plot(x_history[2,:],label='unsat-zone',linewidth=2.0)
plt.plot(x_history[3,:],label='si',linewidth=2.0)
plt.plot(x_history[4,:], label = "sb", linewidth=2.0)
plt.legend()
plt.title("States")
plt.show()

plt.plot(x_history[5,:], label = "shm outflow (z-)")
plt.plot(z_plus, label = "LSTM outflow (z+)")
plt.plot(raw_data_point["y"][0,:,0], label = "Observed outflow")
plt.legend()
plt.title("Outflows")

In [ ]:
error = torch.tensor(x_history[5,:]) - z_plus
mse = (torch.sum(torch.square(error)))/365
print(torch.mean(error))
print(mse)

## Kalman Filter Implementation

$x_{k+1} = M_{k+1}(x_k, \theta, u_{k+1}) + \eta_{k+1}$

$z_{k+1} = H_{k+1}(x_{k+1}, \theta) + \epsilon_{k+1}$

| symbol   | description  | what it is in this case |
| -------- | ------------ | ----------------------- | 
| $x$      | state vector | snow storage, fast flow, unsaturated, interflow, baseflow|
| $M$      | nonlinear model operator | shm equaitons to update states |
| $\theta$ | time invariant model params | shm params. dd, f_thr, sumax etc. |
| $u$ | model input | meterological forcing |
| $z$ | observation vector | true outflow/LSTM outflow |
| $H$ | observation operator | shm equation for outflow |
| $\eta$ | model error | vector length x |
| $Q$ | model error covariance matrix | square size length x |
| $\epsilon$ | observation error | vector length x |
| $R$ | observation error covariance matrix | square size length x |

Note: $ s_{k+1} = s^* - s^*/k = s^*(1 - 1/k)$, so

 $s^* = s_{k+1} (\frac{k}{k-1})$

$$\begin{align*} 
z_{k+1} &= H_{k+1}(x_{k+1}, \theta) + \epsilon_{k+1} \\
 &= \text{qfout + qiout + qbout}\\
 &= sf^*/k_f + si^*/k_i + sb^*/k_b \\
 &= sf_{k+1}(\frac{1}{k_f -1}) + si_{k+1}(\frac{1}{k_i -1}) +sb_{k+1}(\frac{1}{k_b -1})\\
\end{align*}$$

So the Jacobian of $H$ with respect to the states is

$$ \bm{H} = \begin{bmatrix}
0 & \frac{1}{k_f -1} & 0 & \frac{1}{k_i -1} & \frac{1}{k_b -1} 
\end{bmatrix}$$

This is constant in time if the parameters are constant.

Using torch autograd to get linear M and H. Each column is devivative with respect to each state, the first 5 rows are the states and the last row is the outflow.

- Its probably wrong to use the last row of the Jacobian as the linear observation operator $\bm{H}$ becasue it is supposed to be a function of $x_{k+1}$ not $x_k$

In [ ]:
# test autograd for jacobain
test_state = torch.tensor([20.0,30.0,40.0,50.0,60.0])
j = 4
x_d = raw_data_point["x_d"]
pet = get_pet.daily_pet_jensen2016( T_avg=(x_d['tmax(C)'][0, j,0] + x_d['tmin(C)'][0, j,0])/2, S_rad= x_d['srad(W/m2)'][0,j,0],)
x_conceptual_timestep = np.stack([x_d["prcp(mm/day)"][0,j,0], pet, x_d["tmin(C)"][0,j,0], x_d["tmax(C)"][0,j,0]])
x_conceptual_timestep = torch.tensor(x_conceptual_timestep, dtype = torch.float32, device = "cpu").unsqueeze(dim = 0)
shm_tensor = shm.timestep_shm_tensor_fxn(timestep_params=shm_parameters, x_conceptual_timestep=x_conceptual_timestep, device = "cpu")
J = torch.autograd.functional.jacobian(shm_tensor, test_state)
print(J)
print(J.shape)
M = J[0:5,:]
print(M)


In [1]:
# KF
pt = raw_data_point #ordered_data[1]
ys = pt["y"] #[1,365,1]
n = ys.shape[1]
x_d = pt['x_d'] # dict of forcing params mapped to [1,365,1]

# intitialize states for shm, 0, 1, 5, 10, 15 
ss, sf, su, si, sb = shm.initialize_states(batch_size=1, device = 'cpu')
x = torch.tensor([[ss],[sf],[su],[si],[sb]]) 
print(x.shape)
m_history = np.zeros((6,730)) # to plot, = [ss, sf, su, si, sb, z_minus, ]
p_history = np.zeros((6,730)) # = [ss+, sf+, su+, si+, sb+, z_plus, ]

P_p = 10.0*torch.eye(5) # initial error covariance matrix, guess?
Q = 3.0*torch.eye(5) # guess?
R = 10.0 # guess? observation error covariance
H = torch.tensor([[0.0, 1/(shm_parameters["kf"]-1),0.0,1/(shm_parameters["ki"]-1),1/(shm_parameters["kb"]-1)]]) # 1 by 5
for j in range(n): #each day
    pet = get_pet.daily_pet_jensen2016(
                T_avg=(x_d['tmax(C)'][0, j,0] + x_d['tmin(C)'][0, j,0])/2,
                S_rad= x_d['srad(W/m2)'][0,j,0],
            )
    x_conceptual_timestep = np.stack([x_d["prcp(mm/day)"][0,j,0], pet, x_d["tmin(C)"][0,j,0], x_d["tmax(C)"][0,j,0]])
    x_conceptual_timestep = torch.tensor(x_conceptual_timestep, dtype = torch.float32, device = "cpu").unsqueeze(dim = 0) # size [1,4] row vector
    shm_tensor = shm.timestep_shm_tensor_fxn(timestep_params=shm_parameters, x_conceptual_timestep=x_conceptual_timestep, device = "cpu")
    J = torch.autograd.functional.jacobian(shm_tensor, x.squeeze())
    M = J[0:5,:]
    x[0], x[1], x[2], x[3], x[4], z_m = shm.timestep_shm(x[0], x[1], x[2], x[3], x[4], shm_parameters, x_conceptual_timestep, device = x_conceptual_timestep.device)
    m_history[0:5,j] = x[0:5,0]
    m_history[5,j] = z_m
    z_p = z_plus[j] # LSTM outflow
    p_history[5,j] = z_p 
    d = z_p - z_m
    P_m = M @ P_p @ M.T + Q #eq 23)
    K = torch.div(P_m @ H.T , H @ P_m @ H.T + R) # 
    P_p = P_m - K @ H @ P_m #eq 25)
    x = x + K*d # eq 24)
    p_history[0:5, j] = x[0:5,0]



NameError: name 'raw_data_point' is not defined

### To Do
- set up shm for auto grad done
- slice jacobian for M done
- finish kf implementation done?
- test and plot

In [ ]:
#plotting
plt.figure(figsize=[16,10])
plt.plot(p_history[5,:], label = "z_plus", linewidth = 2)
plt.plot(m_history[5,:], label = "z_minus", linewidth = 2)
plt.plot(x_history[5,:], label = "shm outflow no DA", linewidth = 2)
plt.plot(raw_data_point["y"][0,:,0], label = "Observed outflow")
plt.plot()
plt.legend()
plt.show()

plt.figure(figsize = [12,8])
plt.plot(p_history[0,:], label = "ssp")
plt.plot(p_history[1,:], label = "sfp")
plt.plot(p_history[2,:], label = "sup")
plt.plot(p_history[3,:], label = "sip")
plt.plot(p_history[4,:], label = "sbp")
plt.title("State vector +")
plt.legend()
plt.show